# 07 - FRED Macroeconomic Data

## Objective
Download macroeconomic indicators from FRED and convert them into a daily dataset aligned with market trading days.

## Data Source
FRED (Federal Reserve Economic Data) API

## Date Range
2023-01-01 to 2026-01-01

## Indicators
| Indicator | FRED Series ID |
|----------|----------------|
| Interest Rate | FEDFUNDS |
| Inflation (CPI) | CPIAUCSL |
| Unemployment Rate | UNRATE |
| GDP | GDP |

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import warnings

# FRED API library
try:
    from fredapi import Fred
    print('fredapi imported successfully')
except ImportError:
    print('Installing fredapi...')
    import subprocess
    subprocess.check_call(['pip', 'install', 'fredapi', '-q'])
    from fredapi import Fred
    print('fredapi installed and imported')

warnings.filterwarnings('ignore')
print('Libraries imported successfully')

Installing fredapi...
fredapi installed and imported
Libraries imported successfully


## 2. Load API Key

In [2]:
# Load environment variables from .env file
load_dotenv()

# Get FRED API key
FRED_API_KEY = os.getenv('FRED_API_KEY')

if not FRED_API_KEY:
    raise ValueError(
        'FRED_API_KEY not found in environment variables.\n'
        'Please create a .env file with:\n'
        'FRED_API_KEY=your_api_key_here\n\n'
        'Get a free API key at: https://fred.stlouisfed.org/docs/api/api_key.html'
    )

# Initialize FRED client
fred = Fred(api_key=FRED_API_KEY)
print('FRED API client initialized successfully')
print(f'API Key: {FRED_API_KEY[:4]}...{FRED_API_KEY[-4:]}')

FRED API client initialized successfully
API Key: 0a40...02a4


## 3. Configuration

In [3]:
# Date range
START_DATE = '2023-01-01'
END_DATE = '2026-01-01'

# FRED series configuration
FRED_SERIES = {
    'FEDFUNDS': 'Interest_Rate',      # Federal Funds Rate (monthly)
    'CPIAUCSL': 'Inflation',          # CPI All Urban Consumers (monthly)
    'UNRATE': 'Unemployment',         # Unemployment Rate (monthly)
    'GDP': 'GDP'                       # Gross Domestic Product (quarterly)
}

# Output path
OUTPUT_DIR = Path('../Market_Data/raw')
OUTPUT_FILE = OUTPUT_DIR / 'macro_data.csv'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Date range: {START_DATE} to {END_DATE}')
print(f'Series to download: {list(FRED_SERIES.keys())}')
print(f'Output file: {OUTPUT_FILE}')

Date range: 2023-01-01 to 2026-01-01
Series to download: ['FEDFUNDS', 'CPIAUCSL', 'UNRATE', 'GDP']
Output file: ..\Market_Data\raw\macro_data.csv


## 4. Download FRED Data

In [4]:
def download_fred_series(fred_client, series_id, start_date, end_date):
    '''
    Download a single FRED series.
    
    Args:
        fred_client: FRED API client
        series_id: FRED series identifier
        start_date: Start date
        end_date: End date
    
    Returns:
        pandas Series with the data
    '''
    try:
        data = fred_client.get_series(
            series_id,
            observation_start=start_date,
            observation_end=end_date
        )
        print(f'  {series_id}: Downloaded {len(data)} observations')
        return data
    except Exception as e:
        print(f'  {series_id}: Error - {e}')
        return None


def download_all_series(fred_client, series_dict, start_date, end_date):
    '''
    Download all FRED series and combine into DataFrame.
    '''
    print('Downloading FRED series...')
    
    all_data = {}
    for series_id, column_name in series_dict.items():
        data = download_fred_series(fred_client, series_id, start_date, end_date)
        if data is not None:
            all_data[column_name] = data
    
    if not all_data:
        raise ValueError('No data downloaded from FRED')
    
    # Combine into DataFrame
    df = pd.DataFrame(all_data)
    df.index = pd.to_datetime(df.index)
    df.index.name = 'Date'
    
    return df


# Download all series
macro_df = download_all_series(fred, FRED_SERIES, START_DATE, END_DATE)
print(f'\nDownloaded data shape: {macro_df.shape}')

  FEDFUNDS: Downloaded 37 observations
  CPIAUCSL: Downloaded 37 observations
  UNRATE: Downloaded 37 observations
  GDP: Downloaded 12 observations

Downloaded data shape: (37, 4)


In [5]:
# Preview raw data (before daily conversion)
print('Raw FRED Data (Monthly/Quarterly):')
print(macro_df.head(10).to_string())
print(f'\nData frequency info:')
for col in macro_df.columns:
    non_null = macro_df[col].dropna()
    if len(non_null) > 1:
        freq = (non_null.index[1] - non_null.index[0]).days
        print(f'  {col}: ~{freq} days between observations')

Raw FRED Data (Monthly/Quarterly):
            Interest_Rate  Inflation  Unemployment        GDP
Date                                                         
2023-01-01           4.33    300.420           3.5  27216.445
2023-02-01           4.57    301.450           3.6        NaN
2023-03-01           4.65    301.821           3.5        NaN
2023-04-01           4.83    302.845           3.4  27530.055
2023-05-01           5.06    303.334           3.6        NaN
2023-06-01           5.08    304.014           3.6        NaN
2023-07-01           5.12    304.609           3.5  28074.846
2023-08-01           5.33    306.082           3.7        NaN
2023-09-01           5.33    307.276           3.7        NaN
2023-10-01           5.33    307.696           3.9  28424.722

Data frequency info:
  Interest_Rate: ~31 days between observations
  Inflation: ~31 days between observations
  Unemployment: ~31 days between observations
  GDP: ~90 days between observations


## 5. Convert to Daily Frequency

In [6]:
def convert_to_daily(df, start_date, end_date):
    '''
    Convert monthly/quarterly FRED data to daily frequency using forward fill.
    
    Args:
        df: DataFrame with FRED data (monthly/quarterly)
        start_date: Start date
        end_date: End date
    
    Returns:
        DataFrame with daily frequency
    '''
    # Create complete daily date range
    daily_dates = pd.date_range(start=start_date, end=end_date, freq='D')
    daily_df = pd.DataFrame(index=daily_dates)
    daily_df.index.name = 'Date'
    
    # Merge with existing data
    merged = daily_df.join(df, how='left')
    
    # Forward fill to propagate values
    # This assigns each day the most recent available macro value
    filled = merged.ffill()
    
    # Backward fill any remaining NaN at the start
    filled = filled.bfill()
    
    return filled


# Convert to daily frequency
daily_macro_df = convert_to_daily(macro_df, START_DATE, END_DATE)
print(f'Daily data shape: {daily_macro_df.shape}')

Daily data shape: (1097, 4)


In [7]:
# Preview daily data
print('Daily Macro Data Preview:')
print(daily_macro_df.head(15).to_string())

Daily Macro Data Preview:
            Interest_Rate  Inflation  Unemployment        GDP
Date                                                         
2023-01-01           4.33     300.42           3.5  27216.445
2023-01-02           4.33     300.42           3.5  27216.445
2023-01-03           4.33     300.42           3.5  27216.445
2023-01-04           4.33     300.42           3.5  27216.445
2023-01-05           4.33     300.42           3.5  27216.445
2023-01-06           4.33     300.42           3.5  27216.445
2023-01-07           4.33     300.42           3.5  27216.445
2023-01-08           4.33     300.42           3.5  27216.445
2023-01-09           4.33     300.42           3.5  27216.445
2023-01-10           4.33     300.42           3.5  27216.445
2023-01-11           4.33     300.42           3.5  27216.445
2023-01-12           4.33     300.42           3.5  27216.445
2023-01-13           4.33     300.42           3.5  27216.445
2023-01-14           4.33     300.42        

## 6. Data Cleaning

In [8]:
def clean_macro_data(df):
    '''
    Clean and validate macro data.
    '''
    df = df.copy()
    
    # Reset index to make Date a column
    df = df.reset_index()
    
    # Ensure Date is datetime
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Sort by Date
    df = df.sort_values('Date').reset_index(drop=True)
    
    # Check for any remaining missing values
    missing = df.isnull().sum()
    if missing.any():
        print(f'Warning: Missing values found:\n{missing}')
    
    # Ensure correct column order
    columns = ['Date', 'Interest_Rate', 'Inflation', 'Unemployment', 'GDP']
    df = df[columns]
    
    return df


# Clean data
final_df = clean_macro_data(daily_macro_df)
print(f'Cleaned data shape: {final_df.shape}')

Cleaned data shape: (1097, 5)


In [9]:
# Data types
print('Data Types:')
print(final_df.dtypes)

print('\nMissing Values:')
print(final_df.isnull().sum())

Data Types:
Date             datetime64[ns]
Interest_Rate           float64
Inflation               float64
Unemployment            float64
GDP                     float64
dtype: object

Missing Values:
Date             0
Interest_Rate    0
Inflation        0
Unemployment     0
GDP              0
dtype: int64


## 7. Save Output

In [10]:
# Save to CSV
final_df.to_csv(OUTPUT_FILE, index=False)
print(f'Dataset saved to: {OUTPUT_FILE}')
print(f'Records saved: {len(final_df)}')

Dataset saved to: ..\Market_Data\raw\macro_data.csv
Records saved: 1097


## 8. Validation

In [11]:
# Reload and validate
validation_df = pd.read_csv(OUTPUT_FILE, parse_dates=['Date'])

print('=' * 60)
print('VALIDATION REPORT')
print('=' * 60)

# Dataset shape
print(f'\n1. Dataset Shape: {validation_df.shape}')
print(f'   - Rows: {validation_df.shape[0]}')
print(f'   - Columns: {validation_df.shape[1]}')

# Date range
print(f'\n2. Date Range:')
print(f'   - Start: {validation_df["Date"].min()}')
print(f'   - End: {validation_df["Date"].max()}')

# Missing values
print(f'\n3. Missing Values:')
print(validation_df.isnull().sum().to_string())

# Data types
print(f'\n4. Data Types:')
print(validation_df.dtypes.to_string())

# First 5 rows
print(f'\n5. First 5 Rows:')
print(validation_df.head().to_string())

print('\n' + '=' * 60)
print('VALIDATION COMPLETE')
print('=' * 60)

VALIDATION REPORT

1. Dataset Shape: (1097, 5)
   - Rows: 1097
   - Columns: 5

2. Date Range:
   - Start: 2023-01-01 00:00:00
   - End: 2026-01-01 00:00:00

3. Missing Values:
Date             0
Interest_Rate    0
Inflation        0
Unemployment     0
GDP              0

4. Data Types:
Date             datetime64[ns]
Interest_Rate           float64
Inflation               float64
Unemployment            float64
GDP                     float64

5. First 5 Rows:
        Date  Interest_Rate  Inflation  Unemployment        GDP
0 2023-01-01           4.33     300.42           3.5  27216.445
1 2023-01-02           4.33     300.42           3.5  27216.445
2 2023-01-03           4.33     300.42           3.5  27216.445
3 2023-01-04           4.33     300.42           3.5  27216.445
4 2023-01-05           4.33     300.42           3.5  27216.445

VALIDATION COMPLETE


In [12]:
# Statistics
print('\nMacro Data Statistics:')
print(validation_df.describe().round(4).to_string())


Macro Data Statistics:
                      Date  Interest_Rate  Inflation  Unemployment         GDP
count                 1097      1097.0000  1097.0000     1097.0000   1097.0000
mean   2024-07-02 00:00:00         4.7930   313.5428        3.9757  29297.7664
min    2023-01-01 00:00:00         3.6400   300.4200        3.4000  27216.4450
25%    2023-10-02 00:00:00         4.3300   307.6960        3.7000  28424.7220
50%    2024-07-02 00:00:00         4.8300   313.5690        4.1000  29511.6640
75%    2025-04-02 00:00:00         5.3300   320.3020        4.2000  30485.7290
max    2026-01-01 00:00:00         5.3300   326.5880        4.5000  31442.4830
std                    NaN         0.5033     7.5198        0.3033   1299.1014


## Summary

This notebook:

1. **Downloaded FRED Data**
   - Interest Rate (FEDFUNDS) - monthly
   - Inflation/CPI (CPIAUCSL) - monthly
   - Unemployment Rate (UNRATE) - monthly
   - GDP (GDP) - quarterly

2. **Converted to Daily Frequency**
   - Used forward fill to propagate monthly/quarterly values to daily
   - Each trading day now has corresponding macro data

3. **Output**
   - Saved to `Market_Data/raw/macro_data.csv`
   - Columns: Date | Interest_Rate | Inflation | Unemployment | GDP

**Note:** Macro data may need to be lagged when merging with stock data to avoid look-ahead bias.